# Annual single-pass pairwise agreement statistics

Pool exact sufficient sums and mergeable distribution summaries from the v4 daily cache. No full-resolution ERA5 results are read by this notebook.

In [ ]:
from datetime import date
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd
import plotly.graph_objects as go

from liljegren_analysis import (
    histogram_quantile_statistics,
    pooled_sufficient_statistics,
)

In [ ]:
repository = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
database_path = repository / "data" / "daily_pairwise_all_implementations_v4.duckdb"
year = 2025
start_date = date(year, 1, 1)
end_date = date(year, 12, 31)

In [ ]:
with duckdb.connect(str(database_path), read_only=True) as connection:
    metadata = connection.execute("SELECT * FROM pairwise_schema").df()
    daily = connection.execute(
        "SELECT * FROM daily_pairwise WHERE result_date BETWEEN ? AND ? ORDER BY result_date, implementation_a, implementation_b",
        [start_date, end_date],
    ).df()
    histogram = connection.execute(
        """
        SELECT implementation_a, implementation_b, abs_bin_id,
               min(abs_bin_lower_c) AS abs_bin_lower_c,
               min(abs_bin_upper_c) AS abs_bin_upper_c,
               sum(n)::BIGINT AS n,
               fsum(area_weight_sum) AS area_weight_sum
        FROM daily_pairwise_abs_histogram
        WHERE result_date BETWEEN ? AND ?
        GROUP BY implementation_a, implementation_b, abs_bin_id
        ORDER BY implementation_a, implementation_b, abs_bin_id
        """,
        [start_date, end_date],
    ).df()

In [ ]:
if metadata.to_dict("records") != [{
    "schema_version": 4,
    "sample_scope": "all-implementations",
    "implementation_set_version": 1,
    "abs_histogram_version": 1,
}]:
    raise ValueError(f"Unexpected pairwise schema metadata:\n{metadata}")
pair_columns = ["implementation_a", "implementation_b"]
expected_dates = (end_date - start_date).days + 1
coverage = daily.groupby(pair_columns)["result_date"].nunique()
if len(coverage) != 21 or not coverage.eq(expected_dates).all():
    raise ValueError(f"Incomplete daily pair coverage:\n{coverage}")

coverage

In [ ]:
records = []
for pair_key, pair in daily.groupby(pair_columns, sort=True):
    implementation_a, implementation_b = pair_key
    statistics = pooled_sufficient_statistics(
        pair["n"],
        sum_differences=pair["sum_difference_c"],
        sum_squared_differences=pair["sum_squared_difference_c2"],
        sum_absolute_differences=pair["sum_absolute_difference_c"],
        min_differences=pair["min_difference_c"],
        max_differences=pair["max_difference_c"],
        max_absolute_differences=pair["max_absolute_difference_c"],
        area_weight_sums=pair["area_weight_sum"],
        area_weighted_sum_differences=pair["area_weighted_difference_sum_c"],
        area_weighted_sum_squared_differences=pair["area_weighted_squared_difference_sum_c2"],
        area_weighted_sum_absolute_differences=pair["area_weighted_absolute_difference_sum_c"],
        exceedance_counts_0_1=pair["n_absolute_difference_gt_0_1_c"],
        exceedance_counts_0_2=pair["n_absolute_difference_gt_0_2_c"],
        exceedance_counts_0_5=pair["n_absolute_difference_gt_0_5_c"],
        exceedance_area_weights_0_1=pair["area_weight_absolute_difference_gt_0_1_c"],
        exceedance_area_weights_0_2=pair["area_weight_absolute_difference_gt_0_2_c"],
        exceedance_area_weights_0_5=pair["area_weight_absolute_difference_gt_0_5_c"],
    )
    pair_histogram = histogram.loc[
        histogram["implementation_a"].eq(implementation_a)
        & histogram["implementation_b"].eq(implementation_b)
    ]
    quantiles = histogram_quantile_statistics(
        pair_histogram["abs_bin_id"],
        bin_lowers=pair_histogram["abs_bin_lower_c"],
        bin_uppers=pair_histogram["abs_bin_upper_c"],
        counts=pair_histogram["n"],
        area_weight_sums=pair_histogram["area_weight_sum"],
        expected_count=statistics["n"],
        expected_area_weight_sum=statistics["area_weight_sum"],
    )
    records.append({
        "implementation_a": implementation_a,
        "implementation_b": implementation_b,
        **statistics,
        **quantiles,
    })

yearly_pairwise = pd.DataFrame.from_records(records)

yearly_pairwise

## Annual agreement matrices

In [ ]:
metric_specs = {
    "n": {"label": "Matched observations", "unit": "rows"},
    "mean_difference_c": {"label": "Mean difference", "unit": "°C", "signed": True, "reverse_sign": True},
    "stddev_difference_c": {"label": "Difference SD", "unit": "°C"},
    "mean_absolute_difference_c": {"label": "MAE", "unit": "°C"},
    "rmse_c": {"label": "RMSE", "unit": "°C"},
    "loa_lower_c": {"label": "Lower parametric LoA", "unit": "°C", "signed": True, "reverse_sign": True, "reverse_metric": "loa_upper_c"},
    "loa_upper_c": {"label": "Upper parametric LoA", "unit": "°C", "signed": True, "reverse_sign": True, "reverse_metric": "loa_lower_c"},
    "median_absolute_difference_c": {"label": "Approximate P50 |difference|", "unit": "°C"},
    "p95_absolute_difference_c": {"label": "Approximate P95 |difference|", "unit": "°C"},
    "p99_absolute_difference_c": {"label": "Approximate P99 |difference|", "unit": "°C"},
    "max_absolute_difference_c": {"label": "Maximum |difference|", "unit": "°C"},
    "area_weighted_mean_difference_c": {"label": "Area-weighted mean difference", "unit": "°C", "signed": True, "reverse_sign": True},
    "area_weighted_stddev_difference_c": {"label": "Area-weighted difference SD", "unit": "°C"},
    "area_weighted_mean_absolute_difference_c": {"label": "Area-weighted MAE", "unit": "°C"},
    "area_weighted_rmse_c": {"label": "Area-weighted RMSE", "unit": "°C"},
    "area_weighted_loa_lower_c": {"label": "Area-weighted lower parametric LoA", "unit": "°C", "signed": True, "reverse_sign": True, "reverse_metric": "area_weighted_loa_upper_c"},
    "area_weighted_loa_upper_c": {"label": "Area-weighted upper parametric LoA", "unit": "°C", "signed": True, "reverse_sign": True, "reverse_metric": "area_weighted_loa_lower_c"},
    "area_weighted_median_absolute_difference_c": {"label": "Area-weighted approximate P50 |difference|", "unit": "°C"},
    "area_weighted_p95_absolute_difference_c": {"label": "Area-weighted approximate P95 |difference|", "unit": "°C"},
    "area_weighted_p99_absolute_difference_c": {"label": "Area-weighted approximate P99 |difference|", "unit": "°C"},
}

implementations = sorted(set(yearly_pairwise["implementation_a"]) | set(yearly_pairwise["implementation_b"]))
metric_matrices = {metric: pd.DataFrame(np.nan, index=implementations, columns=implementations) for metric in metric_specs}

for row in yearly_pairwise.itertuples(index=False):
    for metric, spec in metric_specs.items():
        value = getattr(row, metric)
        metric_matrices[metric].loc[row.implementation_a, row.implementation_b] = value
        reverse_value = getattr(row, spec.get("reverse_metric", metric))
        metric_matrices[metric].loc[row.implementation_b, row.implementation_a] = -reverse_value if spec.get("reverse_sign") else reverse_value

In [ ]:
figure = go.Figure()
metrics = list(metric_specs)
metric_domains = {}
for metric in metrics:
    counterpart = metric.removeprefix("area_weighted_") if metric.startswith("area_weighted_") else f"area_weighted_{metric}"
    domain_metrics = (metric, counterpart) if counterpart in metric_matrices else (metric,)
    finite = np.concatenate([
        values[np.isfinite(values)]
        for domain_metric in domain_metrics
        for values in [metric_matrices[domain_metric].to_numpy()]
    ])
    minimum, maximum = float(np.min(finite)), float(np.max(finite))
    if minimum == maximum:
        padding = max(abs(minimum) * 1e-12, np.finfo(float).eps)
        minimum, maximum = minimum - padding, maximum + padding
    metric_domains[metric] = (minimum, maximum)

for position, metric in enumerate(metrics):
    spec = metric_specs[metric]
    matrix = metric_matrices[metric]
    minimum, maximum = metric_domains[metric]
    signed = spec.get("signed", False)
    figure.add_trace(go.Heatmap(
        z=matrix.to_numpy(), x=implementations, y=implementations,
        text=matrix.map(lambda value: "" if pd.isna(value) else f"{value:,.3g}").to_numpy(),
        texttemplate="%{text}", colorscale="RdBu_r" if signed else "YlOrRd",
        zmin=minimum, zmax=maximum, zmid=0 if signed else None,
        colorbar={"title": {"text": spec["unit"]}}, visible=position == 0,
        hovertemplate="<b>%{y} − %{x}</b><br>" + spec["label"] + ": %{z:,.5g} " + spec["unit"] + "<extra></extra>",
    ))

figure.update_layout(
    template="plotly_dark",
    height=760, margin={"l": 120, "r": 80, "t": 180, "b": 100},
    updatemenus=[{
        "buttons": [{"label": metric_specs[m]["label"], "method": "update", "args": [{"visible": [candidate == m for candidate in metrics]}]} for m in metrics],
        "direction": "down", "showactive": True,
        "x": 0, "xanchor": "left", "y": 1.08, "yanchor": "bottom",
    }],
    annotations=[{"text": "Metric", "showarrow": False, "x": 0, "xref": "paper", "xanchor": "left", "y": 1.18, "yref": "paper", "yanchor": "bottom"}],
    xaxis={"title": "implementation_b"},
    yaxis={"title": "implementation_a", "autorange": "reversed", "scaleanchor": "x", "scaleratio": 1},
)

figure.show()